# EmpathyTransformer V3 — Training on Kaggle (T4 x2)

Runtime → **GPU T4 x2**

Upload `aitest/` sebagai **Kaggle Dataset** (Add Data → Upload).
Dataset akan ter-mount di `/kaggle/input/aitest/`

In [ ]:
import os, shutil
INPUT_DIR = '/kaggle/input/aitest'
WORK_DIR = '/kaggle/working/aitest'
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
shutil.copytree(INPUT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
print('Files:', [f for f in os.listdir('.') if os.path.isfile(f)])
print('Data:', os.listdir('data') if os.path.isdir('data') else 'NO DATA')

In [ ]:
import torch
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1024**3:.1f}GB')

In [ ]:
!pip install tokenizers tqdm -q
print('Done')

## 1. Train Tokenizer

In [ ]:
import os; os.chdir('/kaggle/working/aitest')
DATA_PATH = 'data/train_mixed.jsonl'
TOK_PATH = 'tokenizer.json'
if not os.path.exists(TOK_PATH):
    !python tokenizer.py --data {DATA_PATH} --vocab-size 16384 --save {TOK_PATH}
    print('Tokenizer created')
else:
    print('Exists')

## 2. Train Model

2× T4 → batch 8 per GPU (DataParallel otomatis). Grad accum 8 → effective 128.
Training ~4-6 jam / 10 epoch.

In [ ]:
import os; os.chdir('/kaggle/working/aitest')

import subprocess

cmd = [
    'python', 'train.py',
    '--data', 'data/train_mixed.jsonl',
    '--tokenizer', 'tokenizer.json',
    '--epochs', '10',
    '--batch-size', '8',
    '--grad-accum', '8',
    '--lr', '1e-3',
    '--amp',
    '--grad-checkpoint',
    '--empathy',
    '--checkpoint-dir', './checkpoints',
    '--log-every', '5',
    '--save-every', '500',
    '--val-every', '250',
]
subprocess.run(cmd)# Ganti jadi --thinking-steps 4 kalo mau depth-thinking

## 3. Test

In [ ]:
%run inference.py --model checkpoints/best.pt --tokenizer tokenizer.json --prompt "I feel lonely today"

## 4. Save ke output

In [ ]:
import shutil, os
OUT = '/kaggle/working/aitest_checkpoints'
os.makedirs(OUT, exist_ok=True)
for f in ['best.pt', 'final.pt']:
    src = f'/kaggle/working/aitest/checkpoints/{f}'
    if os.path.exists(src): shutil.copy2(src, OUT)
if os.path.exists('/kaggle/working/aitest/tokenizer.json'):
    shutil.copy2('/kaggle/working/aitest/tokenizer.json', OUT)
for f in os.listdir(OUT):
    print(f'  {f}: {os.path.getsize(os.path.join(OUT, f))/1024**2:.1f}MB')